# Project 1: Yellow Taxi Trip Analysis Using Hive

**Dataset:** yellow_tripdata_2015-01-06.csv (10,000 records)

**Platform:** Apache Hive on Hadoop

## Step 0: Setup - Create Database, Table, and Load Data

In [ ]:
-- Create and select database
CREATE DATABASE IF NOT EXISTS taxi_analysis;
USE taxi_analysis;

In [ ]:
-- Create table with provided DDL
CREATE TABLE IF NOT EXISTS taxidata (
    vendor_id           STRING,
    pickup_datetime     STRING,
    dropoff_datetime    STRING,
    passenger_count     INT,
    trip_distance       DECIMAL(9,6),
    pickup_longitude    DECIMAL(9,6),
    pickup_latitude     DECIMAL(9,6),
    rate_code           INT,
    store_and_fwd_flag  STRING,
    dropoff_longitude   DECIMAL(9,6),
    dropoff_latitude    DECIMAL(9,6),
    payment_type        STRING,
    fare_amount         DECIMAL(9,6),
    extra               DECIMAL(9,6),
    mta_tax             DECIMAL(9,6),
    tip_amount          DECIMAL(9,6),
    tolls_amount        DECIMAL(9,6),
    total_amount        DECIMAL(9,6),
    trip_time_in_secs   INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
TBLPROPERTIES ("skip.header.line.count"="1");

In [ ]:
-- Load data from CSV
LOAD DATA LOCAL INPATH '/opt/hive/data/csv/yellow_tripdata_2015-01-06.csv'
INTO TABLE taxidata;

In [ ]:
-- Verify data loaded correctly
SELECT * FROM taxidata LIMIT 10;

DESCRIBE taxidata;

SELECT COUNT(*) FROM taxidata;

## Question 1: Total Number of Trips

In [ ]:
SELECT COUNT(*) AS total_trips
FROM taxidata;

-- Result: 10000

## Question 2: Total Revenue Generated

In [ ]:
SELECT ROUND(SUM(total_amount), 2) AS total_revenue
FROM taxidata;

-- Result: 160546.81

## Question 2a: Fraction of Revenue Paid as Tolls

In [ ]:
SELECT
    ROUND(SUM(tolls_amount), 2)                                AS total_tolls,
    ROUND(SUM(total_amount), 2)                                AS total_revenue,
    ROUND(SUM(tolls_amount) / SUM(total_amount) * 100, 2)     AS toll_fraction_pct
FROM taxidata;

-- Result: Tolls represent approximately 1.56% of total revenue

## Question 2b: Fraction of Revenue That Is Driver Tips

In [ ]:
SELECT
    ROUND(SUM(tip_amount), 2)                                  AS total_tips,
    ROUND(SUM(total_amount), 2)                                AS total_revenue,
    ROUND(SUM(tip_amount) / SUM(total_amount) * 100, 2)       AS tip_fraction_pct
FROM taxidata;

-- Result: Tips represent approximately 10.79% of total revenue

## Question 3: Average Trip Amount

In [ ]:
SELECT ROUND(AVG(total_amount), 2) AS avg_trip_amount
FROM taxidata;

-- Result: 16.05

## Question 4: Average Trip Distance

In [ ]:
SELECT ROUND(AVG(trip_distance), 2) AS avg_trip_distance
FROM taxidata;

-- Result: 3.25 miles

## Question 5: Distinct Payment Types

In [ ]:
SELECT COUNT(DISTINCT payment_type) AS distinct_payment_types
FROM taxidata;

-- Result: 4 types

-- Show what they are (1=Credit Card, 2=Cash, 3=No Charge, 4=Dispute)
SELECT DISTINCT payment_type
FROM taxidata
ORDER BY payment_type;

## Question 6: Averages by Payment Type

For each payment type: average fare, average tip, average tax

In [ ]:
SELECT
    payment_type,
    ROUND(AVG(fare_amount), 2)  AS avg_fare,
    ROUND(AVG(tip_amount), 2)   AS avg_tip,
    ROUND(AVG(mta_tax), 2)      AS avg_tax
FROM taxidata
GROUP BY payment_type
ORDER BY payment_type;

-- Results:
-- Type 1 (Credit Card): avg_fare=13.56, avg_tip=2.70, avg_tax=0.50
-- Type 2 (Cash):        avg_fare=11.39, avg_tip=0.00, avg_tax=0.50
-- Type 3 (No Charge):   avg_fare=13.21, avg_tip=0.00, avg_tax=0.42
-- Type 4 (Dispute):     avg_fare=12.22, avg_tip=0.00, avg_tax=0.50
-- Note: Cash tips are not recorded electronically, so Type 2 shows $0.00

## Question 7: Highest Revenue Hour

In [ ]:
SELECT
    HOUR(pickup_datetime)               AS pickup_hour,
    ROUND(AVG(total_amount), 2)         AS avg_revenue
FROM taxidata
GROUP BY HOUR(pickup_datetime)
ORDER BY avg_revenue DESC;

-- Answer: Hour 22 (10 PM) generates the highest average revenue at ~$16.24 per trip

In [ ]:
-- Top hour only
SELECT pickup_hour, avg_revenue
FROM (
    SELECT
        HOUR(pickup_datetime)               AS pickup_hour,
        ROUND(AVG(total_amount), 2)         AS avg_revenue
    FROM taxidata
    GROUP BY HOUR(pickup_datetime)
) hourly_revenue
ORDER BY avg_revenue DESC
LIMIT 1;

-- Result: Hour 22 (10 PM), avg revenue ~$16.24

## Key Observations

1. **Credit card users tip significantly more** ($2.70 avg) compared to all other payment types ($0.00), partly because cash tips are not recorded in the dataset.

2. **Tolls represent a small fraction** (1.56%) of total revenue, while tips account for roughly 10.79%.

3. **Late evening hours (10-11 PM)** generate the highest per-trip revenue, likely due to longer trips (airport runs, outer borough travel) and surge/extra charges.

4. **Average trip is 3.25 miles** costing roughly $16.05, consistent with short urban taxi rides in Manhattan.